In [2]:
import os
import numpy as np
import pandas as pd
import scipy.signal as ss
import mne
import antropy as ant

from mne.decoding import CSP

In [3]:
df = pd.read_csv('dataset_BCI_completo.csv')

df.head()

,Sujeto,Condicion,Pot_Mu_C3,Pot_Mu_C4,Pot_Mu_Cz,Pot_Beta_C3,Pot_Beta_C4,Pot_Beta_Cz,Coh_Mu_C3_C4,Coh_Mu_C3_Cz,Coh_Mu_C4_Cz,Coh_Beta_C3_C4,Coh_Beta_C3_Cz,Coh_Beta_C4_Cz,Tipo
0,S001,Derecha,3.306863e-11,2.853744e-11,3.415199e-11,8.826270e-12,8.399120e-12,9.569256e-12,0.607774,0.842584,0.820079,0.550151,0.826825,0.683419,Imaginación Motora
1,S001,Reposo,3.105931e-11,2.168272e-11,3.053431e-11,1.279435e-11,7.969746e-12,1.130518e-11,0.702683,0.817302,0.850840,0.669249,0.849468,0.817461,Imaginación Motora
2,S001,Izquierda,3.244346e-11,2.711707e-11,3.483905e-11,8.441985e-12,6.591918e-12,7.897998e-12,0.736120,0.829747,0.922998,0.548569,0.725583,0.784959,Imaginación Motora
3,S001,Reposo,3.012843e-11,2.871449e-11,3.223302e-11,8.385304e-12,7.612996e-12,8.640117e-12,0.524534,0.806225,0.752072,0.567207,0.786638,0.717071,Imaginación Motora
4,S001,Izquierda,3.225722e-11,1.758506e-11,2.746542e-11,9.115796e-12,7.481623e-12,9.094855e-12,0.760350,0.895360,0.838313,0.546283,0.792682,0.728047,Imaginación Motora


In [4]:
ruta_base='Sujetos'

In [5]:
def obtener_metricas_eeg(
        raw,
        canales=['C3','Cz','C4']
    ):

    """Calcula métricas relevantes para EEG:
    - PSD (Densidad Espectral de Potencia)
    - ERD/ERS (Desconexión/Reconexión Relativa)
    - Entropía Espectral
    - Parámetros de Hjorth
    - Coherencia entre pares de canales
    - Lateralización (diferencia entre C3 y C4)"""

    raw = raw.copy() # Crear una copia para no modificar el original

    raw.pick(canales) # Seleccionar solo los canales de interés 

    Fs = raw.info['sfreq'] # Frecuencia de muestreo

    datos = raw.get_data()

    resultados={} 

    # PSD + MU + BETA + ERD/ERS (Proyecto 1)
    # Entropía Espectral + Hjorth (Proyecto 2)
    for i,canal in enumerate(canales):

        señal=datos[i]

        # Welch para PSD
        f,Pxx = ss.welch(

            señal,

            fs=Fs,

            window='hann',

            nperseg=int(2*Fs)
        )
        
        # Mascara de frecuencias para bandas mu y beta
        idx_mu=(f>=8)&(f<=13)
        idx_beta=(f>=14)&(f<=30)

        # Promedio de potencia en bandas mu y beta
        mu=np.mean(Pxx[idx_mu])
        beta=np.mean(Pxx[idx_beta])
        # Potencia total 
        total=np.mean(Pxx)
        
        # ERD
        ERD_mu=((mu-total)/total)*100
        ERD_beta=((beta-total)/total)*100

        # ENTROPÍA ESPECTRAL (Proyecto 2)
        entropy = ant.spectral_entropy(

            señal,

            sf=Fs,

            method='welch',

            normalize=True
        )

        # HJORTH (Proyecto 2)
        actividad=np.var(señal)

        movilidad,complejidad=ant.hjorth_params(
            señal
        )

        resultados[canal]={

            'PSD':Pxx,
            'frecuencias':f,

            'mu':mu,
            'beta':beta,

            'ERD_mu':ERD_mu,
            'ERD_beta':ERD_beta,

            'spectral_entropy':entropy,

            'hjorth_activity':actividad,
            'hjorth_mobility':movilidad,
            'hjorth_complexity':complejidad
        }

    # COHERENCIA
    pares=[

        ('C3','C4'),
        ('C3','Cz'),
        ('C4','Cz')

    ]

    coherencias={}

    for c1,c2 in pares:

        idx1=canales.index(c1)
        idx2=canales.index(c2)

        f_coh,Cxy = ss.coherence(

            datos[idx1],

            datos[idx2],

            fs=Fs,

            nperseg=int(2*Fs)
        )

        coh_mu=np.mean(
            Cxy[(f_coh>=8)&(f_coh<=13)]
        )

        coh_beta=np.mean(
            Cxy[(f_coh>=14)&(f_coh<=30)]
        )

        coherencias[f'{c1}-{c2}']={

            'coh_mu':coh_mu,
            'coh_beta':coh_beta
        }

    resultados['coherencia']=coherencias

    # LATERALIZACIÓN
    resultados['lateralizacion']={

        'mu_C3_C4':

        resultados['C3']['mu']
        -
        resultados['C4']['mu'],

        'beta_C3_C4':

        resultados['C3']['beta']
        -
        resultados['C4']['beta']
    }

    return resultados

In [6]:
def calcular_CSP(
        X,
        y,
        n_components=2
    ):

    """
    X = [epochs, canales, muestras]

    y = etiquetas
    """

    csp=CSP(

        n_components=n_components,

        log=True,

        norm_trace=False
    )

    features=csp.fit_transform(
        X,
        y
    )

    return features,csp

In [7]:
archivo='Sujetos/S001/S001R04.edf'

raw = mne.io.read_raw_edf(

        archivo,

        preload=True,

        verbose=False
)

mne.datasets.eegbci.standardize(raw)

raw.notch_filter(60)

raw.filter(8,30)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge: 60.65 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 60.90 Hz)
- Filter length: 1057 samples (6.606 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge

<RawEDF | S001R04.edf, 64 x 20000 (125.0 s), ~9.8 MiB, data loaded>

In [8]:
metricas = obtener_metricas_eeg(raw)
metricas['C3']['spectral_entropy']

metricas['C3']['hjorth_activity']

metricas['C3']['hjorth_mobility']

metricas['C3']['hjorth_complexity']

np.float64(1.3096563468573343)

In [9]:
metricas['C3']['mu']

metricas['C3']['ERD_mu']

metricas['coherencia']['C3-C4']['coh_mu']

np.float64(0.5411316617615861)

In [10]:
fila={

'sujeto':'sub001',

'tarea':1,

# PROYECTO 1
'mu_C3':metricas['C3']['mu'],
'mu_Cz':metricas['Cz']['mu'],
'mu_C4':metricas['C4']['mu'],

'beta_C3':metricas['C3']['beta'],
'beta_Cz':metricas['Cz']['beta'],
'beta_C4':metricas['C4']['beta'],

'ERD_mu_C3':metricas['C3']['ERD_mu'],

'coh_mu_C3_C4':

metricas['coherencia']['C3-C4']['coh_mu'],

# PROYECTO 2
'entropy_C3':
metricas['C3']['spectral_entropy'],

'entropy_Cz':
metricas['Cz']['spectral_entropy'],

'entropy_C4':
metricas['C4']['spectral_entropy'],

'hjorth_activity_C3':
metricas['C3']['hjorth_activity'],

'hjorth_mobility_C3':
metricas['C3']['hjorth_mobility'],

'hjorth_complexity_C3':
metricas['C3']['hjorth_complexity']

}

### Automatización

Este bloque recorrer automáticamente todos los sujetos y archivos EEG, extraer características y las almacena en un único DataFrame.


In [11]:

# LISTA GENERAL DONDE SE GUARDARÁN TODOS LOS REGITROS

dataset_total = []

In [16]:
# ==========================================================================================
# RECORRER TODOS LOS SUJETOS DE LA BASE
# ==========================================================================================

for sujeto in os.listdir(ruta_base):

    ruta_sujeto = os.path.join(ruta_base, sujeto)

    # Verificar que sea carpeta
    if not os.path.isdir(ruta_sujeto):
        continue

    print(f'Procesando sujeto: {sujeto}')


    # ======================================================================================
    # RECORRER TODOS LOS ARCHIVOS EDF DEL SUJETO
    # ======================================================================================

    for archivo in os.listdir(ruta_sujeto):

        # Seleccionar únicamente archivos EDF
        if not archivo.endswith('.edf'):
            continue

        ruta_archivo = os.path.join(ruta_sujeto, archivo)

        print(f'   Archivo: {archivo}')


        # ==================================================================================
        # CARGAR SEÑAL EEG
        # ==================================================================================

        raw = mne.io.read_raw_edf(

            ruta_archivo,

            preload=True,

            verbose=False
        )

        # Estandarizar nombres de canales
        mne.datasets.eegbci.standardize(raw)


        # ==================================================================================
        # PREPROCESAMIENTO
        # ==================================================================================

        # Eliminar ruido eléctrico
        raw.notch_filter(60, verbose=False)

        # Filtrar bandas motoras
        raw.filter(8, 30, verbose=False)

        # Seleccionar canales motores
        raw.pick(['C3', 'Cz', 'C4'])


        # ==================================================================================
        # EXTRAER EVENTOS
        # ==================================================================================

        eventos, eventos_dict = mne.events_from_annotations(

            raw,

            verbose=False
        )


        # ==================================================================================
        # MAPEAR ETIQUETAS DEL DATASET
        # ==================================================================================

        mapeo = {}

        for clave, valor in eventos_dict.items():

            if clave == 'T0':
                mapeo['Reposo'] = valor

            elif clave == 'T1':
                mapeo['Izquierda'] = valor

            elif clave == 'T2':
                mapeo['Derecha'] = valor


        # ==================================================================================
        # SEGMENTAR EN EPOCHS
        # ==================================================================================

        epochs = mne.Epochs(

            raw,

            eventos,

            event_id=mapeo,

            tmin=0,

            tmax=4,

            baseline=None,

            preload=True,

            verbose=False
        )


        # ==================================================================================
        # RECORRER CADA EPOCH
        # ==================================================================================

        # Obtener todos los epochs como arreglo numpy
        datos_epochs = epochs.get_data()

        # Recorrer cada epoch
        for i in range(len(datos_epochs)):

            # Extraer un epoch individual
            # Forma: [canales x muestras]
            datos_epoch = datos_epochs[i]


            # ==========================================================================
            # CREAR OBJETO TEMPORAL PARA USAR LA FUNCIÓN DEL PUNTO 1
            # ==========================================================================

            info = mne.create_info(

                ch_names=['C3', 'Cz', 'C4'],

                sfreq=raw.info['sfreq'],

                ch_types='eeg'
            )

            raw_epoch = mne.io.RawArray(

                datos_epoch,

                info,

                verbose=False
            )


            # ==========================================================================
            # EXTRAER MÉTRICAS
            # ==========================================================================

            metricas = obtener_metricas_eeg(raw_epoch)


            # ==========================================================================
            # IDENTIFICAR CONDICIÓN DEL EPOCH
            # ==========================================================================

            etiqueta_num = epochs.events[i, -1]

            condicion = [

                k for k, v in mapeo.items()

                if v == etiqueta_num

            ][0]


            # ==========================================================================
            # CONSTRUIR FILA DEL DATASET
            # ==========================================================================

            fila = {

                'Sujeto': sujeto,

                'Archivo': archivo,

                'Condicion': condicion,


                # ================= PSD μ =================

                'Mu_C3':
                    metricas['C3']['mu'],

                'Mu_Cz':
                    metricas['Cz']['mu'],

                'Mu_C4':
                    metricas['C4']['mu'],


                # ================= PSD β =================

                'Beta_C3':
                    metricas['C3']['beta'],

                'Beta_Cz':
                    metricas['Cz']['beta'],

                'Beta_C4':
                    metricas['C4']['beta'],


                # ================= ERD =================

                'ERD_Mu_C3':
                    metricas['C3']['ERD_mu'],

                'ERD_Mu_C4':
                    metricas['C4']['ERD_mu'],


                # ================= ENTROPÍA =================

                'Entropy_C3':
                    metricas['C3']['spectral_entropy'],

                'Entropy_Cz':
                    metricas['Cz']['spectral_entropy'],

                'Entropy_C4':
                    metricas['C4']['spectral_entropy'],


                # ================= HJORTH =================

                'Hjorth_Activity_C3':
                    metricas['C3']['hjorth_activity'],

                'Hjorth_Mobility_C3':
                    metricas['C3']['hjorth_mobility'],

                'Hjorth_Complexity_C3':
                    metricas['C3']['hjorth_complexity'],


                # ================= COHERENCIA =================

                'Coh_Mu_C3_C4':
                    metricas['coherencia']['C3-C4']['coh_mu'],

                'Coh_Beta_C3_C4':
                    metricas['coherencia']['C3-C4']['coh_beta']
            }


            # ==========================================================================
            # AGREGAR FILA AL DATASET GENERAL
            # ==========================================================================

            dataset_total.append(fila)


Procesando sujeto: S001
   Archivo: S001R03.edf
   Archivo: S001R04.edf
   Archivo: S001R07.edf
   Archivo: S001R08.edf
   Archivo: S001R11.edf
   Archivo: S001R12.edf
Procesando sujeto: S002
   Archivo: S002R03.edf
   Archivo: S002R04.edf
   Archivo: S002R07.edf
   Archivo: S002R08.edf
   Archivo: S002R11.edf
   Archivo: S002R12.edf
Procesando sujeto: S003
   Archivo: S003R03.edf
   Archivo: S003R04.edf
   Archivo: S003R07.edf
   Archivo: S003R08.edf
   Archivo: S003R11.edf
   Archivo: S003R12.edf
Procesando sujeto: S004
   Archivo: S004R03.edf
   Archivo: S004R04.edf
   Archivo: S004R07.edf
   Archivo: S004R08.edf
   Archivo: S004R11.edf
   Archivo: S004R12.edf
Procesando sujeto: S005
   Archivo: S005R03.edf
   Archivo: S005R04.edf
   Archivo: S005R07.edf
   Archivo: S005R08.edf
   Archivo: S005R11.edf
   Archivo: S005R12.edf
Procesando sujeto: S006
   Archivo: S006R03.edf
   Archivo: S006R04.edf
   Archivo: S006R07.edf
   Archivo: S006R08.edf
   Archivo: S006R11.edf
   Archivo: S006

C:\Users\MARIANA\AppData\Local\Temp\ipykernel_19280\4067478424.py:35: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(


   Archivo: S100R04.edf


C:\Users\MARIANA\AppData\Local\Temp\ipykernel_19280\4067478424.py:35: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(


   Archivo: S100R07.edf


...

   Archivo: S100R08.edf


C:\Users\MARIANA\AppData\Local\Temp\ipykernel_19280\4067478424.py:35: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(


   Archivo: S100R11.edf


C:\Users\MARIANA\AppData\Local\Temp\ipykernel_19280\4067478424.py:35: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(
C:\Users\MARIANA\AppData\Local\Temp\ipykernel_19280\4067478424.py:35: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(


   Archivo: S100R12.edf
Procesando sujeto: S101
   Archivo: S101R03.edf
   Archivo: S101R04.edf
   Archivo: S101R07.edf
   Archivo: S101R08.edf
   Archivo: S101R11.edf
   Archivo: S101R12.edf
Procesando sujeto: S102
   Archivo: S102R03.edf
   Archivo: S102R04.edf
   Archivo: S102R07.edf
   Archivo: S102R08.edf
   Archivo: S102R11.edf
   Archivo: S102R12.edf
Procesando sujeto: S103
   Archivo: S103R03.edf
   Archivo: S103R04.edf
   Archivo: S103R07.edf
   Archivo: S103R08.edf
   Archivo: S103R11.edf
   Archivo: S103R12.edf
Procesando sujeto: S104
   Archivo: S104R03.edf
   Archivo: S104R04.edf
   Archivo: S104R07.edf
   Archivo: S104R08.edf
   Archivo: S104R11.edf
   Archivo: S104R12.edf
Procesando sujeto: S105
   Archivo: S105R03.edf
   Archivo: S105R04.edf
   Archivo: S105R07.edf
   Archivo: S105R08.edf
   Archivo: S105R11.edf
   Archivo: S105R12.edf
Procesando sujeto: S106
   Archivo: S106R03.edf
   Archivo: S106R04.edf
   Archivo: S106R07.edf
   Archivo: S106R08.edf
   Archivo: S106

In [17]:
# ==========================================================================================
# CONVERTIR TODO A DATAFRAME
# ==========================================================================================
df_dataset = pd.DataFrame(dataset_total)

# ==========================================================================================
# VISUALIZAR RESULTADOS
# ==========================================================================================
print(df_dataset.head())
print('\nDimensiones del dataset:')
print(df_dataset.shape)


  Sujeto      Archivo  Condicion         Mu_C3         Mu_Cz         Mu_C4  \
0   S001  S001R03.edf     Reposo  2.882461e-11  2.795643e-11  1.880007e-11   
1   S001  S001R03.edf    Derecha  2.683128e-11  2.200586e-11  1.342307e-11   
2   S001  S001R03.edf     Reposo  3.055682e-11  2.854258e-11  1.712095e-11   
3   S001  S001R03.edf  Izquierda  3.614583e-11  2.825319e-11  1.582837e-11   
4   S001  S001R03.edf     Reposo  4.194921e-11  4.360823e-11  2.996402e-11   

        Beta_C3       Beta_Cz       Beta_C4   ERD_Mu_C3   ERD_Mu_C4  \
0  7.007411e-12  8.118984e-12  7.218073e-12  613.334450  481.509252   
1  1.152903e-11  1.021584e-11  6.514260e-12  465.339115  438.551677   
2  9.792817e-12  9.536298e-12  7.434085e-12  573.833756  459.564923   
3  8.499539e-12  8.851952e-12  7.567938e-12  692.661769  451.465575   
4  1.238507e-11  1.011724e-11  7.768881e-12  522.642703  597.185078   

   Entropy_C3  Entropy_Cz  Entropy_C4  Hjorth_Activity_C3  Hjorth_Mobility_C3  \
0    0.688693    0.6978

In [18]:
# ==========================================================================================
# GUARDAR DATASET FINAL
# ==========================================================================================

df_dataset.to_csv(

    'dataset_BCI_completo_proyecto2.csv',

    index=False
)

print('\nDataset guardado correctamente.')


Dataset guardado correctamente.


In [19]:
df = pd.read_csv('dataset_BCI_completo_proyecto2.csv')

df.head()

,Sujeto,Archivo,Condicion,Mu_C3,Mu_Cz,Mu_C4,Beta_C3,Beta_Cz,Beta_C4,ERD_Mu_C3,ERD_Mu_C4,Entropy_C3,Entropy_Cz,Entropy_C4,Hjorth_Activity_C3,Hjorth_Mobility_C3,Hjorth_Complexity_C3,Coh_Mu_C3_C4,Coh_Beta_C3_C4
0,S001,S001R03.edf,Reposo,2.882461e-11,2.795643e-11,1.880007e-11,7.007411e-12,8.118984e-12,7.218073e-12,613.334450,481.509252,0.688693,0.697868,0.707952,3.375341e-10,0.591492,1.343486,0.657766,0.588770
1,S001,S001R03.edf,Derecha,2.683128e-11,2.200586e-11,1.342307e-11,1.152903e-11,1.021584e-11,6.514260e-12,465.339115,438.551677,0.700328,0.723544,0.732144,3.801286e-10,0.612639,1.273345,0.722648,0.736650
2,S001,S001R03.edf,Reposo,3.055682e-11,2.854258e-11,1.712095e-11,9.792817e-12,9.536298e-12,7.434085e-12,573.833756,459.564923,0.704148,0.704003,0.717119,3.947629e-10,0.599488,1.360701,0.726484,0.571752
3,S001,S001R03.edf,Izquierda,3.614583e-11,2.825319e-11,1.582837e-11,8.499539e-12,8.851952e-12,7.567938e-12,692.661769,451.465575,0.677741,0.698752,0.721812,3.553030e-10,0.597685,1.299446,0.805119,0.598702
4,S001,S001R03.edf,Reposo,4.194921e-11,4.360823e-11,2.996402e-11,1.238507e-11,1.011724e-11,7.768881e-12,522.642703,597.185078,0.683826,0.674937,0.686692,5.823957e-10,0.583292,1.325214,0.740297,0.571581
